This notebook will take:

- PRL_ready_final.csv
- baptism_family.csv
- marriage_couples.csv
- burial_family.csv

and create the Splink-ready enriched file:

- PRL_ready_final_with_family_evidence.csv

Imports and paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import unicodedata
from IPython.display import display

In [2]:
PRL_FILE = "PRL_ready_personas_final.csv"

FAMILY_DIR = Path("outputs/family_evidence")
OUTPUT_DIR = Path("outputs/splink_ready")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BAPTISM_FAMILY_FILE = FAMILY_DIR / "baptism_family.csv"
MARRIAGE_COUPLES_FILE = FAMILY_DIR / "marriage_couples.csv"
BURIAL_FAMILY_FILE = FAMILY_DIR / "burial_family.csv"

print("Family evidence folder:", FAMILY_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())

Family evidence folder: C:\Users\samav\OneDrive\Documents\Vamsi Documents\Masters in CS notes\GRA\PRL_Splink\outputs\family_evidence
Output folder: C:\Users\samav\OneDrive\Documents\Vamsi Documents\Masters in CS notes\GRA\PRL_Splink\outputs\splink_ready


Helper functions

In [3]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFD", value)
    value = "".join(ch for ch in value if unicodedata.category(ch) != "Mn")
    value = re.sub(r"[^a-z0-9ñ\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    
    return value if value else np.nan


def first_token(value):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    return str(value).strip().split()[0]


def last_token(value):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    
    parts = str(value).strip().split()
    return parts[-1] if len(parts) > 1 else np.nan


def make_key(*values):
    clean_values = []
    
    for value in values:
        if pd.isna(value) or str(value).strip() == "":
            return np.nan
        clean_values.append(str(value).strip())
    
    return " | ".join(clean_values)


def standardize_source_table(value):
    value = normalize_text(value)
    
    if pd.isna(value):
        return np.nan
    
    if value in ["bautismos", "baptisms", "baptism"]:
        return "baptism"
    
    if value in ["matrimonios", "marriages", "marriage"]:
        return "marriage"
    
    if value in ["entierros", "burials", "burial"]:
        return "burial"
    
    return value


def standardize_persona_type(value):
    value = normalize_text(value)
    
    if pd.isna(value):
        return np.nan
    
    role_map = {
        "baptized": "child",
        "baptised": "child",
        "child": "child",
        "bautizado": "child",
        "bautizada": "child",

        "father": "father",
        "padre": "father",

        "mother": "mother",
        "madre": "mother",

        "husband": "husband",
        "groom": "husband",
        "esposo": "husband",
        "marido": "husband",

        "wife": "wife",
        "bride": "wife",
        "esposa": "wife",
        "mujer": "wife",

        "deceased": "deceased",
        "dead": "deceased",
        "difunto": "deceased",
        "difunta": "deceased",

        "father of husband": "husband_father",
        "father_of_husband": "husband_father",
        "husband father": "husband_father",

        "mother of husband": "husband_mother",
        "mother_of_husband": "husband_mother",
        "husband mother": "husband_mother",

        "father of wife": "wife_father",
        "father_of_wife": "wife_father",
        "wife father": "wife_father",

        "mother of wife": "wife_mother",
        "mother_of_wife": "wife_mother",
        "wife mother": "wife_mother",
    }
    
    return role_map.get(value, value)


def add_prefix_except_event_id(df, prefix):
    """
    Adds prefix to all columns except event_idno.
    Example:
    child_name -> baptism_child_name
    """
    rename_map = {}
    
    for col in df.columns:
        if col != "event_idno":
            rename_map[col] = f"{prefix}{col}"
    
    return df.rename(columns=rename_map)

These functions keep the notebook robust. The important function here is:

`add_prefix_except_event_id()`

It prevents column name conflicts when we merge baptism, marriage, and burial tables into PRL.

Load Files

In [4]:
prl = pd.read_csv(PRL_FILE)
baptism_family = pd.read_csv(BAPTISM_FAMILY_FILE)
marriage_couples = pd.read_csv(MARRIAGE_COUPLES_FILE)
burial_family = pd.read_csv(BURIAL_FAMILY_FILE)

print("PRL:", prl.shape)
print("BaptismFamily:", baptism_family.shape)
print("MarriageCouples:", marriage_couples.shape)
print("BurialFamily:", burial_family.shape)

display(prl.head())

C:\Users\samav\AppData\Local\Temp\ipykernel_24212\2145004258.py:1: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  prl = pd.read_csv(PRL_FILE)


PRL: (47072, 32)
BaptismFamily: (6338, 25)
MarriageCouples: (1718, 36)
BurialFamily: (2115, 39)


,persona_idno,event_idno,original_identifier,source_event_type,source_table,event_type,persona_type,role_group,name,lastname,...,death_year,death_place_clean,resident_in_clean,gender,block_lastname_initial,block_lastname_firstname,block_last4_firstname,block_lastname_birthyear,matching_evidence_level,cleaning_issue_flags
0,persona-1,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,baptized,main_life_event_person,domingo,ayquipa,...,NaN,unknown,unknown,male,ayquipa_d,ayquipa_domingo,ayqu_domingo,ayquipa_1790,high,no_major_issue
1,persona-2,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,father,family_relation,lucas,ayquipa,...,NaN,unknown,unknown,male,ayquipa_l,ayquipa_lucas,ayqu_lucas,NaN,medium,no_major_issue
2,persona-3,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,mother,family_relation,sevastiana,quispe,...,NaN,unknown,unknown,female,quispe_s,quispe_sevastiana,quis_sevastiana,NaN,medium,no_major_issue
3,persona-4,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,godfather,godparent,vicente,guamani,...,NaN,unknown,unknown,male,guamani_v,guamani_vicente,guam_vicente,NaN,medium,no_major_issue
4,persona-5,bautizo-2,APAucará-LB-L001_B002,Baptism,bautismos,Bautizo,baptized,main_life_event_person,dominga,lulia,...,NaN,unknown,unknown,female,lulia_d,lulia_dominga,luli_dominga,lulia_1790,high,no_major_issue


Clean and standardize PRL

In [5]:
required_columns = [
    "event_idno",
    "source_table",
    "persona_type",
    "persona_idno",
    "full_name_clean"
]

missing = [col for col in required_columns if col not in prl.columns]

if missing:
    raise ValueError(f"Missing required columns in PRL file: {missing}")

if "event_year" not in prl.columns:
    prl["event_year"] = np.nan

if "event_date" not in prl.columns:
    prl["event_date"] = np.nan

if "event_place_clean" not in prl.columns:
    prl["event_place_clean"] = np.nan

# Standardize important fields
prl["event_idno"] = prl["event_idno"].astype(str).str.strip()
prl["persona_idno"] = prl["persona_idno"].astype(str).str.strip()
prl.loc[prl["persona_idno"].isin(["nan", "None", ""]), "persona_idno"] = np.nan

prl["full_name_clean"] = prl["full_name_clean"].apply(normalize_text)
prl["source_standard"] = prl["source_table"].apply(standardize_source_table)
prl["role_standard"] = prl["persona_type"].apply(standardize_persona_type)
prl["event_year"] = pd.to_numeric(prl["event_year"], errors="coerce")

prl["person_first_name"] = prl["full_name_clean"].apply(first_token)
prl["person_last_name"] = prl["full_name_clean"].apply(last_token)

print("Standardized source counts:")
display(prl["source_standard"].value_counts(dropna=False))

print("Standardized role counts:")
display(prl["role_standard"].value_counts(dropna=False))

Standardized source counts:


source_standard
baptism     24986
marriage    16691
burial       5395
Name: count, dtype: int64

Standardized role counts:


role_standard
mother            7614
father            7369
child             6340
witness           4249
godparent         3260
godmother         3251
godfather         3012
deceased          2120
wife              2060
husband           2051
wife_mother       1459
husband_mother    1439
wife_father       1438
husband_father    1410
Name: count, dtype: int64

Prepare family tables with prefixes

In [6]:
# Clean event_idno in family tables
for df in [baptism_family, marriage_couples, burial_family]:
    df["event_idno"] = df["event_idno"].astype(str).str.strip()

# Add prefixes to avoid column conflicts
baptism_prefixed = add_prefix_except_event_id(baptism_family, "baptism_")
marriage_prefixed = add_prefix_except_event_id(marriage_couples, "marriage_")
burial_prefixed = add_prefix_except_event_id(burial_family, "burial_")

print("Baptism prefixed columns:")
print(baptism_prefixed.columns.tolist()[:20])

print("\nMarriage prefixed columns:")
print(marriage_prefixed.columns.tolist()[:20])

print("\nBurial prefixed columns:")
print(burial_prefixed.columns.tolist()[:20])

Baptism prefixed columns:
['event_idno', 'baptism_event_date', 'baptism_event_year', 'baptism_event_place_clean', 'baptism_child_name', 'baptism_father_name', 'baptism_mother_name', 'baptism_child_persona_id', 'baptism_father_persona_id', 'baptism_mother_persona_id', 'baptism_child_first_name', 'baptism_child_last_name', 'baptism_father_first_name', 'baptism_father_last_name', 'baptism_mother_first_name', 'baptism_mother_last_name', 'baptism_parent_pair_key', 'baptism_child_parent_key', 'baptism_father_motherfirst_key', 'baptism_mother_fatherfirst_key']

Marriage prefixed columns:
['event_idno', 'marriage_event_date', 'marriage_event_year', 'marriage_event_place_clean', 'marriage_husband_name', 'marriage_husband_father_name', 'marriage_husband_mother_name', 'marriage_wife_name', 'marriage_wife_father_name', 'marriage_wife_mother_name', 'marriage_husband_persona_id', 'marriage_husband_father_persona_id', 'marriage_husband_mother_persona_id', 'marriage_wife_persona_id', 'marriage_wife_fa

After prefixing, columns become:

`baptism_child_name`
`baptism_father_name`
`baptism_mother_name`

`marriage_husband_name`
`marriage_wife_name`

`burial_deceased_name`
`burial_father_name`
`burial_mother_name`

This makes the final PRL file clear and avoids duplicate column names.

Merge family evidence into PRL

In [7]:
prl_enriched = prl.copy()

prl_enriched = prl_enriched.merge(
    baptism_prefixed,
    on="event_idno",
    how="left"
)

prl_enriched = prl_enriched.merge(
    marriage_prefixed,
    on="event_idno",
    how="left"
)

prl_enriched = prl_enriched.merge(
    burial_prefixed,
    on="event_idno",
    how="left"
)

print("Original PRL shape:", prl.shape)
print("Enriched PRL shape:", prl_enriched.shape)

display(prl_enriched.head())

Original PRL shape: (47072, 36)
Enriched PRL shape: (47072, 133)


,persona_idno,event_idno,original_identifier,source_event_type,source_table,event_type,persona_type,role_group,name,lastname,...,burial_father_motherfirst_key,burial_mother_fatherfirst_key,burial_has_deceased,burial_has_father,burial_has_mother,burial_has_husband,burial_has_wife,burial_has_spouse,burial_has_deceased_parent_key,burial_has_deceased_spouse_key
0,persona-1,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,baptized,main_life_event_person,domingo,ayquipa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,persona-2,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,father,family_relation,lucas,ayquipa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,persona-3,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,mother,family_relation,sevastiana,quispe,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,persona-4,bautizo-1,APAucará-LB-L001_B001,Baptism,bautismos,Bautizo,godfather,godparent,vicente,guamani,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,persona-5,bautizo-2,APAucará-LB-L001_B002,Baptism,bautismos,Bautizo,baptized,main_life_event_person,dominga,lulia,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Add evidence availability flags

In [8]:
# Baptism flags
prl_enriched["has_baptism_context"] = prl_enriched["source_standard"].eq("baptism")
prl_enriched["has_baptism_parent_pair"] = prl_enriched.get(
    "baptism_parent_pair_key", pd.Series(index=prl_enriched.index)
).notna()

prl_enriched["has_baptism_child_parent_key"] = prl_enriched.get(
    "baptism_child_parent_key", pd.Series(index=prl_enriched.index)
).notna()

# Marriage flags
prl_enriched["has_marriage_context"] = prl_enriched["source_standard"].eq("marriage")
prl_enriched["has_marriage_spouse_pair"] = prl_enriched.get(
    "marriage_spouse_pair_key", pd.Series(index=prl_enriched.index)
).notna()

# Burial flags
prl_enriched["has_burial_context"] = prl_enriched["source_standard"].eq("burial")
prl_enriched["has_burial_deceased_parent_key"] = prl_enriched.get(
    "burial_deceased_parent_key", pd.Series(index=prl_enriched.index)
).notna()

prl_enriched["has_burial_deceased_spouse_key"] = prl_enriched.get(
    "burial_deceased_spouse_key", pd.Series(index=prl_enriched.index)
).notna()

flag_cols = [
    "has_baptism_context",
    "has_baptism_parent_pair",
    "has_baptism_child_parent_key",
    "has_marriage_context",
    "has_marriage_spouse_pair",
    "has_burial_context",
    "has_burial_deceased_parent_key",
    "has_burial_deceased_spouse_key"
]

display(prl_enriched[flag_cols].mean().sort_values(ascending=False))

has_baptism_context               0.530804
has_baptism_parent_pair           0.512831
has_baptism_child_parent_key      0.512831
has_marriage_context              0.354584
has_marriage_spouse_pair          0.354075
has_burial_context                0.114612
has_burial_deceased_parent_key    0.088057
has_burial_deceased_spouse_key    0.042297
dtype: float64

Create universal context columns

In [9]:
def get_context_partner(row):
    """
    Returns the person's partner/context person depending on source and role.
    """
    source = row.get("source_standard")
    role = row.get("role_standard")
    
    # Baptism parent context
    if source == "baptism":
        if role == "father":
            return row.get("baptism_mother_name")
        if role == "mother":
            return row.get("baptism_father_name")
        if role == "child":
            return np.nan
    
    # Marriage spouse context
    if source == "marriage":
        if role == "husband":
            return row.get("marriage_wife_name")
        if role == "wife":
            return row.get("marriage_husband_name")
    
    # Burial spouse context
    if source == "burial":
        if role == "deceased":
            return row.get("burial_spouse_name")
        if role == "husband":
            return row.get("burial_deceased_name")
        if role == "wife":
            return row.get("burial_deceased_name")
    
    return np.nan


def get_context_father(row):
    source = row.get("source_standard")
    role = row.get("role_standard")
    
    if source == "baptism":
        return row.get("baptism_father_name")
    
    if source == "burial":
        return row.get("burial_father_name")
    
    if source == "marriage":
        if role == "husband":
            return row.get("marriage_husband_father_name")
        if role == "wife":
            return row.get("marriage_wife_father_name")
    
    return np.nan


def get_context_mother(row):
    source = row.get("source_standard")
    role = row.get("role_standard")
    
    if source == "baptism":
        return row.get("baptism_mother_name")
    
    if source == "burial":
        return row.get("burial_mother_name")
    
    if source == "marriage":
        if role == "husband":
            return row.get("marriage_husband_mother_name")
        if role == "wife":
            return row.get("marriage_wife_mother_name")
    
    return np.nan


prl_enriched["context_partner_name"] = prl_enriched.apply(get_context_partner, axis=1)
prl_enriched["context_father_name"] = prl_enriched.apply(get_context_father, axis=1)
prl_enriched["context_mother_name"] = prl_enriched.apply(get_context_mother, axis=1)

prl_enriched["context_parent_pair_key"] = prl_enriched.apply(
    lambda x: make_key(x.get("context_father_name"), x.get("context_mother_name")),
    axis=1
)

prl_enriched["context_partner_first_name"] = prl_enriched["context_partner_name"].apply(first_token)
prl_enriched["context_partner_last_name"] = prl_enriched["context_partner_name"].apply(last_token)

prl_enriched["context_father_first_name"] = prl_enriched["context_father_name"].apply(first_token)
prl_enriched["context_father_last_name"] = prl_enriched["context_father_name"].apply(last_token)

prl_enriched["context_mother_first_name"] = prl_enriched["context_mother_name"].apply(first_token)
prl_enriched["context_mother_last_name"] = prl_enriched["context_mother_name"].apply(last_token)

display(prl_enriched[[
    "source_standard",
    "role_standard",
    "full_name_clean",
    "context_partner_name",
    "context_father_name",
    "context_mother_name",
    "context_parent_pair_key"
]].head(20))

,source_standard,role_standard,full_name_clean,context_partner_name,context_father_name,context_mother_name,context_parent_pair_key
0,baptism,child,domingo ayquipa,NaN,lucas ayquipa,sevastiana quispe,lucas ayquipa | sevastiana quispe
1,baptism,father,lucas ayquipa,sevastiana quispe,lucas ayquipa,sevastiana quispe,lucas ayquipa | sevastiana quispe
2,baptism,mother,sevastiana quispe,lucas ayquipa,lucas ayquipa,sevastiana quispe,lucas ayquipa | sevastiana quispe
3,baptism,godfather,vicente guamani,NaN,lucas ayquipa,sevastiana quispe,lucas ayquipa | sevastiana quispe
4,baptism,child,dominga lulia,NaN,juan lulia,jospha gomes,juan lulia | jospha gomes
5,baptism,father,juan lulia,jospha gomes,juan lulia,jospha gomes,juan lulia | jospha gomes
6,baptism,mother,jospha gomes,juan lulia,juan lulia,jospha gomes,juan lulia | jospha gomes
7,baptism,godfather,ignacio varientos,NaN,juan lulia,jospha gomes,juan lulia | jospha gomes
8,baptism,child,bartola quispe,NaN,jacinto quispe,juliana chinchay,jacinto quispe | juliana chinchay
9,baptism,father,jacinto quispe,juliana chinchay,jacinto quispe,juliana chinchay,jacinto quispe | juliana chinchay


Create Splink-ready baptized child file

In [10]:
splink_baptized_child = prl_enriched[
    (prl_enriched["source_standard"] == "baptism") &
    (prl_enriched["role_standard"] == "child")
].copy()

# Keep useful columns
baptized_child_cols = [
    "persona_idno",
    "event_idno",
    "source_table",
    "source_standard",
    "persona_type",
    "role_standard",
    "full_name_clean",
    "person_first_name",
    "person_last_name",
    "event_date",
    "event_year",
    "event_place_clean",
    "baptism_child_name",
    "baptism_father_name",
    "baptism_mother_name",
    "baptism_child_parent_key",
    "baptism_parent_pair_key",
    "baptism_father_motherfirst_key",
    "baptism_mother_fatherfirst_key",
    "context_father_name",
    "context_mother_name",
    "context_parent_pair_key"
]

baptized_child_cols = [col for col in baptized_child_cols if col in splink_baptized_child.columns]
splink_baptized_child = splink_baptized_child[baptized_child_cols].copy()

print("Splink baptized child file:", splink_baptized_child.shape)
display(splink_baptized_child.head())

Splink baptized child file: (6340, 22)


,persona_idno,event_idno,source_table,source_standard,persona_type,role_standard,full_name_clean,person_first_name,person_last_name,event_date,...,baptism_child_name,baptism_father_name,baptism_mother_name,baptism_child_parent_key,baptism_parent_pair_key,baptism_father_motherfirst_key,baptism_mother_fatherfirst_key,context_father_name,context_mother_name,context_parent_pair_key
0,persona-1,bautizo-1,bautismos,baptism,baptized,child,domingo ayquipa,domingo,ayquipa,1790-10-04,...,domingo ayquipa,lucas ayquipa,sevastiana quispe,domingo ayquipa | lucas ayquipa | sevastiana q...,lucas ayquipa | sevastiana quispe,lucas ayquipa | sevastiana,sevastiana quispe | lucas,lucas ayquipa,sevastiana quispe,lucas ayquipa | sevastiana quispe
4,persona-5,bautizo-2,bautismos,baptism,baptized,child,dominga lulia,dominga,lulia,1790-10-06,...,dominga lulia,juan lulia,jospha gomes,dominga lulia | juan lulia | jospha gomes,juan lulia | jospha gomes,juan lulia | jospha,jospha gomes | juan,juan lulia,jospha gomes,juan lulia | jospha gomes
8,persona-9,bautizo-3,bautismos,baptism,baptized,child,bartola quispe,bartola,quispe,1790-10-07,...,bartola quispe,jacinto quispe,juliana chinchay,bartola quispe | jacinto quispe | juliana chin...,jacinto quispe | juliana chinchay,jacinto quispe | juliana,juliana chinchay | jacinto,jacinto quispe,juliana chinchay,jacinto quispe | juliana chinchay
12,persona-13,bautizo-4,bautismos,baptism,baptized,child,francisca cuebas,francisca,cuebas,1790-10-20,...,francisca cuebas,juan cuebas,clemenzia manco,francisca cuebas | juan cuebas | clemenzia manco,juan cuebas | clemenzia manco,juan cuebas | clemenzia,clemenzia manco | juan,juan cuebas,clemenzia manco,juan cuebas | clemenzia manco
16,persona-17,bautizo-5,bautismos,baptism,baptized,child,pedro manxo,pedro,manxo,1790-10-20,...,pedro manxo,santos manxo,baleriana arango,pedro manxo | santos manxo | baleriana arango,santos manxo | baleriana arango,santos manxo | baleriana,baleriana arango | santos,santos manxo,baleriana arango,santos manxo | baleriana arango


Create Splink-ready baptism parent files

In [11]:
splink_baptism_fathers = prl_enriched[
    (prl_enriched["source_standard"] == "baptism") &
    (prl_enriched["role_standard"] == "father")
].copy()

splink_baptism_mothers = prl_enriched[
    (prl_enriched["source_standard"] == "baptism") &
    (prl_enriched["role_standard"] == "mother")
].copy()

parent_cols = [
    "persona_idno",
    "event_idno",
    "source_table",
    "source_standard",
    "persona_type",
    "role_standard",
    "full_name_clean",
    "person_first_name",
    "person_last_name",
    "event_date",
    "event_year",
    "event_place_clean",
    "baptism_child_name",
    "baptism_father_name",
    "baptism_mother_name",
    "baptism_parent_pair_key",
    "context_partner_name",
    "context_partner_first_name",
    "context_partner_last_name",
    "context_parent_pair_key"
]

parent_cols = [col for col in parent_cols if col in prl_enriched.columns]

splink_baptism_fathers = splink_baptism_fathers[parent_cols].copy()
splink_baptism_mothers = splink_baptism_mothers[parent_cols].copy()

print("Baptism fathers:", splink_baptism_fathers.shape)
print("Baptism mothers:", splink_baptism_mothers.shape)

display(splink_baptism_fathers.head())
display(splink_baptism_mothers.head())

Baptism fathers: (6070, 20)
Baptism mothers: (6313, 20)


,persona_idno,event_idno,source_table,source_standard,persona_type,role_standard,full_name_clean,person_first_name,person_last_name,event_date,event_year,event_place_clean,baptism_child_name,baptism_father_name,baptism_mother_name,baptism_parent_pair_key,context_partner_name,context_partner_first_name,context_partner_last_name,context_parent_pair_key
1,persona-2,bautizo-1,bautismos,baptism,father,father,lucas ayquipa,lucas,ayquipa,1790-10-04,1790.0,pampamarca,domingo ayquipa,lucas ayquipa,sevastiana quispe,lucas ayquipa | sevastiana quispe,sevastiana quispe,sevastiana,quispe,lucas ayquipa | sevastiana quispe
5,persona-6,bautizo-2,bautismos,baptism,father,father,juan lulia,juan,lulia,1790-10-06,1790.0,pampamarca,dominga lulia,juan lulia,jospha gomes,juan lulia | jospha gomes,jospha gomes,jospha,gomes,juan lulia | jospha gomes
9,persona-10,bautizo-3,bautismos,baptism,father,father,jacinto quispe,jacinto,quispe,1790-10-07,1790.0,pampamarca,bartola quispe,jacinto quispe,juliana chinchay,jacinto quispe | juliana chinchay,juliana chinchay,juliana,chinchay,jacinto quispe | juliana chinchay
13,persona-14,bautizo-4,bautismos,baptism,father,father,juan cuebas,juan,cuebas,1790-10-20,1790.0,aucara,francisca cuebas,juan cuebas,clemenzia manco,juan cuebas | clemenzia manco,clemenzia manco,clemenzia,manco,juan cuebas | clemenzia manco
17,persona-18,bautizo-5,bautismos,baptism,father,father,santos manxo,santos,manxo,1790-10-20,1790.0,aucara,pedro manxo,santos manxo,baleriana arango,santos manxo | baleriana arango,baleriana arango,baleriana,arango,santos manxo | baleriana arango


,persona_idno,event_idno,source_table,source_standard,persona_type,role_standard,full_name_clean,person_first_name,person_last_name,event_date,event_year,event_place_clean,baptism_child_name,baptism_father_name,baptism_mother_name,baptism_parent_pair_key,context_partner_name,context_partner_first_name,context_partner_last_name,context_parent_pair_key
2,persona-3,bautizo-1,bautismos,baptism,mother,mother,sevastiana quispe,sevastiana,quispe,1790-10-04,1790.0,pampamarca,domingo ayquipa,lucas ayquipa,sevastiana quispe,lucas ayquipa | sevastiana quispe,lucas ayquipa,lucas,ayquipa,lucas ayquipa | sevastiana quispe
6,persona-7,bautizo-2,bautismos,baptism,mother,mother,jospha gomes,jospha,gomes,1790-10-06,1790.0,pampamarca,dominga lulia,juan lulia,jospha gomes,juan lulia | jospha gomes,juan lulia,juan,lulia,juan lulia | jospha gomes
10,persona-11,bautizo-3,bautismos,baptism,mother,mother,juliana chinchay,juliana,chinchay,1790-10-07,1790.0,pampamarca,bartola quispe,jacinto quispe,juliana chinchay,jacinto quispe | juliana chinchay,jacinto quispe,jacinto,quispe,jacinto quispe | juliana chinchay
14,persona-15,bautizo-4,bautismos,baptism,mother,mother,clemenzia manco,clemenzia,manco,1790-10-20,1790.0,aucara,francisca cuebas,juan cuebas,clemenzia manco,juan cuebas | clemenzia manco,juan cuebas,juan,cuebas,juan cuebas | clemenzia manco
18,persona-19,bautizo-5,bautismos,baptism,mother,mother,baleriana arango,baleriana,arango,1790-10-20,1790.0,aucara,pedro manxo,santos manxo,baleriana arango,santos manxo | baleriana arango,santos manxo,santos,manxo,santos manxo | baleriana arango


Create Splink-ready marriage spouse file

In [12]:
splink_marriage_spouses = prl_enriched[
    (prl_enriched["source_standard"] == "marriage") &
    (prl_enriched["role_standard"].isin(["husband", "wife"]))
].copy()

marriage_spouse_cols = [
    "persona_idno",
    "event_idno",
    "source_table",
    "source_standard",
    "persona_type",
    "role_standard",
    "full_name_clean",
    "person_first_name",
    "person_last_name",
    "event_date",
    "event_year",
    "event_place_clean",
    "marriage_husband_name",
    "marriage_wife_name",
    "marriage_spouse_pair_key",
    "marriage_husband_wifefirst_key",
    "marriage_wife_husbandfirst_key",
    "context_partner_name",
    "context_partner_first_name",
    "context_partner_last_name",
    "context_father_name",
    "context_mother_name",
    "context_parent_pair_key"
]

marriage_spouse_cols = [col for col in marriage_spouse_cols if col in splink_marriage_spouses.columns]
splink_marriage_spouses = splink_marriage_spouses[marriage_spouse_cols].copy()

print("Marriage spouses:", splink_marriage_spouses.shape)
display(splink_marriage_spouses.head())

Marriage spouses: (3436, 23)


,persona_idno,event_idno,source_table,source_standard,persona_type,role_standard,full_name_clean,person_first_name,person_last_name,event_date,...,marriage_wife_name,marriage_spouse_pair_key,marriage_husband_wifefirst_key,marriage_wife_husbandfirst_key,context_partner_name,context_partner_first_name,context_partner_last_name,context_father_name,context_mother_name,context_parent_pair_key
24986,persona-24987,matrimonio-1,matrimonios,marriage,husband,husband,jose manl manuel de la roca,jose,roca,1816-12-06,...,juana rodrigues,jose manl manuel de la roca | juana rodrigues,jose manl manuel de la roca | juana,juana rodrigues | jose,juana rodrigues,juana,rodrigues,acencio roca,leonor guerrero,acencio roca | leonor guerrero
24987,persona-24988,matrimonio-1,matrimonios,marriage,wife,wife,juana rodrigues,juana,rodrigues,1816-12-06,...,juana rodrigues,jose manl manuel de la roca | juana rodrigues,jose manl manuel de la roca | juana,juana rodrigues | jose,jose manl manuel de la roca,jose,roca,pedro rodrigues,magdalena sotelo,pedro rodrigues | magdalena sotelo
24997,persona-24998,matrimonio-2,matrimonios,marriage,husband,husband,esteban castillo,esteban,castillo,1816-12-12,...,ambrocia tasqui,esteban castillo | ambrocia tasqui,esteban castillo | ambrocia,ambrocia tasqui | esteban,ambrocia tasqui,ambrocia,tasqui,matheo castillo,ma maria torres,matheo castillo | ma maria torres
24998,persona-24999,matrimonio-2,matrimonios,marriage,wife,wife,ambrocia tasqui,ambrocia,tasqui,1816-12-12,...,ambrocia tasqui,esteban castillo | ambrocia tasqui,esteban castillo | ambrocia,ambrocia tasqui | esteban,esteban castillo,esteban,castillo,pedro tasqui,ma maria palomino,pedro tasqui | ma maria palomino
25008,persona-25009,matrimonio-3,matrimonios,marriage,husband,husband,alexandro ramires,alexandro,ramires,1817-03-05,...,sipriana coillo,alexandro ramires | sipriana coillo,alexandro ramires | sipriana,sipriana coillo | alexandro,sipriana coillo,sipriana,coillo,leonor romani,franca francisca paucar,leonor romani | franca francisca paucar


Create Splink-ready burial file

In [13]:
splink_burial_personas = prl_enriched[
    prl_enriched["source_standard"] == "burial"
].copy()

burial_cols = [
    "persona_idno",
    "event_idno",
    "source_table",
    "source_standard",
    "persona_type",
    "role_standard",
    "full_name_clean",
    "person_first_name",
    "person_last_name",
    "event_date",
    "event_year",
    "event_place_clean",
    "burial_deceased_name",
    "burial_father_name",
    "burial_mother_name",
    "burial_husband_name",
    "burial_wife_name",
    "burial_spouse_name",
    "burial_deceased_parent_key",
    "burial_deceased_spouse_key",
    "context_partner_name",
    "context_father_name",
    "context_mother_name",
    "context_parent_pair_key"
]

burial_cols = [col for col in burial_cols if col in splink_burial_personas.columns]
splink_burial_personas = splink_burial_personas[burial_cols].copy()

print("Burial personas:", splink_burial_personas.shape)
display(splink_burial_personas.head())

Burial personas: (5395, 24)


,persona_idno,event_idno,source_table,source_standard,persona_type,role_standard,full_name_clean,person_first_name,person_last_name,event_date,...,burial_mother_name,burial_husband_name,burial_wife_name,burial_spouse_name,burial_deceased_parent_key,burial_deceased_spouse_key,context_partner_name,context_father_name,context_mother_name,context_parent_pair_key
41677,persona-41678,entierro-1,entierros,burial,deceased,deceased,julian xavies,julian,xavies,1846-10-06,...,NaN,NaN,mercedes lupa,mercedes lupa,NaN,julian xavies | mercedes lupa,mercedes lupa,NaN,NaN,NaN
41678,persona-41679,entierro-1,entierros,burial,wife,wife,mercedes lupa,mercedes,lupa,1846-10-06,...,NaN,NaN,mercedes lupa,mercedes lupa,NaN,julian xavies | mercedes lupa,julian xavies,NaN,NaN,NaN
41679,persona-41680,entierro-2,entierros,burial,deceased,deceased,joce raime,joce,raime,1846-10-07,...,NaN,NaN,francisca cucho,francisca cucho,NaN,joce raime | francisca cucho,francisca cucho,NaN,NaN,NaN
41680,persona-41681,entierro-2,entierros,burial,wife,wife,francisca cucho,francisca,cucho,1846-10-07,...,NaN,NaN,francisca cucho,francisca cucho,NaN,joce raime | francisca cucho,joce raime,NaN,NaN,NaN
41681,persona-41682,entierro-3,entierros,burial,deceased,deceased,martina condori,martina,condori,1846-11-02,...,NaN,luciano ccoyllo,NaN,luciano ccoyllo,NaN,martina condori | luciano ccoyllo,luciano ccoyllo,NaN,NaN,NaN


Create enrichment report

In [14]:
def coverage_summary(df, table_name):
    rows = []
    
    for col in df.columns:
        total_count = len(df)
        non_null_count = df[col].notna().sum()
        non_null_pct = non_null_count / total_count if total_count > 0 else 0
        
        rows.append({
            "table": table_name,
            "column": col,
            "non_null_count": non_null_count,
            "total_count": total_count,
            "non_null_pct": round(non_null_pct, 4)
        })
    
    return pd.DataFrame(rows)


enrichment_summary = pd.DataFrame([
    {
        "dataset": "PRL original",
        "rows": len(prl),
        "columns": prl.shape[1]
    },
    {
        "dataset": "PRL enriched",
        "rows": len(prl_enriched),
        "columns": prl_enriched.shape[1]
    },
    {
        "dataset": "Splink baptized child",
        "rows": len(splink_baptized_child),
        "columns": splink_baptized_child.shape[1]
    },
    {
        "dataset": "Splink baptism fathers",
        "rows": len(splink_baptism_fathers),
        "columns": splink_baptism_fathers.shape[1]
    },
    {
        "dataset": "Splink baptism mothers",
        "rows": len(splink_baptism_mothers),
        "columns": splink_baptism_mothers.shape[1]
    },
    {
        "dataset": "Splink marriage spouses",
        "rows": len(splink_marriage_spouses),
        "columns": splink_marriage_spouses.shape[1]
    },
    {
        "dataset": "Splink burial personas",
        "rows": len(splink_burial_personas),
        "columns": splink_burial_personas.shape[1]
    }
])

context_coverage = pd.concat([
    coverage_summary(prl_enriched[[
        col for col in [
            "has_baptism_parent_pair",
            "has_baptism_child_parent_key",
            "has_marriage_spouse_pair",
            "has_burial_deceased_parent_key",
            "has_burial_deceased_spouse_key",
            "context_partner_name",
            "context_father_name",
            "context_mother_name",
            "context_parent_pair_key"
        ] if col in prl_enriched.columns
    ]], "PRL Enriched Context Flags")
], ignore_index=True)

display(enrichment_summary)
display(context_coverage)

,dataset,rows,columns
0,PRL original,47072,36
1,PRL enriched,47072,151
2,Splink baptized child,6340,22
3,Splink baptism fathers,6070,20
4,Splink baptism mothers,6313,20
5,Splink marriage spouses,3436,23
6,Splink burial personas,5395,24


,table,column,non_null_count,total_count,non_null_pct
0,PRL Enriched Context Flags,has_baptism_parent_pair,47072,47072,1.0000
1,PRL Enriched Context Flags,has_baptism_child_parent_key,47072,47072,1.0000
2,PRL Enriched Context Flags,has_marriage_spouse_pair,47072,47072,1.0000
3,PRL Enriched Context Flags,has_burial_deceased_parent_key,47072,47072,1.0000
4,PRL Enriched Context Flags,has_burial_deceased_spouse_key,47072,47072,1.0000
5,PRL Enriched Context Flags,context_partner_name,16878,47072,0.3586
6,PRL Enriched Context Flags,context_father_name,31235,47072,0.6636
7,PRL Enriched Context Flags,context_mother_name,32004,47072,0.6799
8,PRL Enriched Context Flags,context_parent_pair_key,31120,47072,0.6611


In [15]:
# Output paths
prl_enriched_path = OUTPUT_DIR / "PRL_ready_final_with_family_evidence.csv"

baptized_child_path = OUTPUT_DIR / "splink_baptized_child_dedupe.csv"
baptism_fathers_path = OUTPUT_DIR / "splink_baptism_father_matching.csv"
baptism_mothers_path = OUTPUT_DIR / "splink_baptism_mother_matching.csv"
marriage_spouses_path = OUTPUT_DIR / "splink_marriage_spouse_matching.csv"
burial_personas_path = OUTPUT_DIR / "splink_burial_personas.csv"

report_path = OUTPUT_DIR / "family_evidence_enrichment_report.xlsx"

# Save CSVs
prl_enriched.to_csv(prl_enriched_path, index=False, encoding="utf-8-sig")
splink_baptized_child.to_csv(baptized_child_path, index=False, encoding="utf-8-sig")
splink_baptism_fathers.to_csv(baptism_fathers_path, index=False, encoding="utf-8-sig")
splink_baptism_mothers.to_csv(baptism_mothers_path, index=False, encoding="utf-8-sig")
splink_marriage_spouses.to_csv(marriage_spouses_path, index=False, encoding="utf-8-sig")
splink_burial_personas.to_csv(burial_personas_path, index=False, encoding="utf-8-sig")

# Save report
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    enrichment_summary.to_excel(writer, sheet_name="Dataset_Summary", index=False)
    context_coverage.to_excel(writer, sheet_name="Context_Coverage", index=False)
    prl_enriched.head(100).to_excel(writer, sheet_name="PRL_Enriched_Sample", index=False)
    splink_baptized_child.head(100).to_excel(writer, sheet_name="BaptizedChild_Sample", index=False)
    splink_baptism_fathers.head(100).to_excel(writer, sheet_name="FatherMatch_Sample", index=False)
    splink_baptism_mothers.head(100).to_excel(writer, sheet_name="MotherMatch_Sample", index=False)
    splink_marriage_spouses.head(100).to_excel(writer, sheet_name="MarriageSpouse_Sample", index=False)
    splink_burial_personas.head(100).to_excel(writer, sheet_name="BurialPersonas_Sample", index=False)

print("Saved outputs:")
print(prl_enriched_path)
print(baptized_child_path)
print(baptism_fathers_path)
print(baptism_mothers_path)
print(marriage_spouses_path)
print(burial_personas_path)
print(report_path)

Saved outputs:
outputs\splink_ready\PRL_ready_final_with_family_evidence.csv
outputs\splink_ready\splink_baptized_child_dedupe.csv
outputs\splink_ready\splink_baptism_father_matching.csv
outputs\splink_ready\splink_baptism_mother_matching.csv
outputs\splink_ready\splink_marriage_spouse_matching.csv
outputs\splink_ready\splink_burial_personas.csv
outputs\splink_ready\family_evidence_enrichment_report.xlsx
